# ML-09 — Validation and Research Claim Audit

**Capstone lane:** Refresh / Content Opportunity Scoring  
**Decision window:** March 2026  
**Outcome window:** April 2026  
**Model:** Logistic Regression (Week-5 / ML-08)  
**Primary evaluation:** Precision@K (K = 10, 50, 100, 500)

This audit deliberately keeps the established Week-5 data contract and model family, then tests whether the original row-level split gives an optimistic estimate when the same clients can appear on both sides of validation. The honest comparison below uses client-grouped, stratified out-of-fold evaluation.

> **Public-safe rule:** results are described as observed/measured/directional/decision-support. No client names, URLs, or private queries are printed.

## 1. Two paper findings + my methodology questions

### Finding 1 — reported Precision@50 lift
The research paper reports that its learned content-refresh approach outperforms the transparent stale-content baseline, including a reported **1.35× lift in Precision@50**.

**My methodology question:** How exactly is the positive label constructed for the pages entering the Precision@50 evaluation, and is that label defined only from information after the decision point? I would also want to confirm that the baseline and learned model are evaluated on the same held-out population and that the Precision@50 queue is not tuned using the evaluation labels.

**Why this is constructive:** A precise label lineage and untouched evaluation set would make the reported lift much easier for a reader to interpret as measured evidence rather than as a general guarantee.

### Finding 2 — feature importance / predictive signal
The paper discusses search-performance variables such as impressions, average position and CTR as useful signals for identifying content that may need attention.

**My methodology question:** Does the validation design demonstrate that these signals generalize beyond the same clients/content represented during training? In particular, if rows from one client can occur in both train and test, could client-specific patterns make feature importance or ranking performance look stronger than it would for an unseen client?

**Why this is constructive:** A grouped-client evaluation would directly test the broader generalization question without changing the business problem or claiming the paper is incorrect.

**Scope note:** These are methodology questions, not accusations. The purpose of this section is to practice the same standard of scrutiny on my own work.

## 2. My model under an honest split (before/after)

The Week-5 model used a fixed stratified 80/20 row split. Because `client_hash_id` repeats across many rows, the original split can place the same client in both train and test. That is useful as the **before** reference, but it is not the strongest test of generalization to unseen clients.

The **after** evaluation below uses `StratifiedGroupKFold`, grouping by `client_hash_id`. Every row receives an out-of-fold prediction from a model that was trained without that row's client. This preserves the lane while making the validation question materially harder.

April is used only to construct the outcome label. Predictive features are March-only. IDs are grouping/audit fields, never model features.

In [ ]:
%pip -q install duckdb pandas numpy scikit-learn

import os
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is missing. Add your Read token to Colab Secrets as HF_TOKEN.')

con = duckdb.connect()
con.execute('INSTALL httpfs;')
con.execute('LOAD httpfs;')
con.execute('CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)', [HF_TOKEN])
MARCH_REL = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
APRIL_REL = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
RANDOM_STATE = 42
FEATURES = ['march_impressions','march_clicks','march_ctr_pct','march_avg_position','march_impression_days']
TARGET = 'future_decline_label'
GROUP = 'client_hash_id'
KS = [10, 50, 100, 500]

In [ ]:
march_sql = f'''
SELECT client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS march_impressions,
       SUM(gsc_clicks) AS march_clicks,
       CASE WHEN SUM(gsc_impressions) > 0 THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions) ELSE NULL END AS march_ctr_pct,
       CASE WHEN SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END) > 0
            THEN SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0 THEN gsc_impressions * gsc_avg_position ELSE 0 END)
                 / SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0 THEN gsc_impressions ELSE 0 END)
            ELSE NULL END AS march_avg_position,
       COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS march_impression_days
FROM read_parquet('{MARCH_REL}')
WHERE month = '2026-03' AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
'''
april_sql = f'''
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS april_impressions
FROM read_parquet('{APRIL_REL}')
WHERE month = '2026-04' AND gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
'''
march = con.execute(march_sql).df()
april = con.execute(april_sql).df()
key_cols = ['client_hash_id','content_hash_id']
assert not march.duplicated(key_cols).any()
assert not april.duplicated(key_cols).any()
df = march.merge(april, on=key_cols, how='inner', validate='one_to_one')
df[TARGET] = ((df['march_impressions'] > 0) & (df['april_impressions'] < 0.80 * df['march_impressions'])).astype(int)
baseline_band = df.loc[(df['march_impressions'] >= 500) & df['march_avg_position'].between(4,20,inclusive='both') & df['march_ctr_pct'].notna()]
ctr_cutoff = baseline_band['march_ctr_pct'].median()
assert pd.notna(ctr_cutoff)
print(f'Rows: {len(df):,}')
print(f'Clients: {df[GROUP].nunique():,}')
print(f'Observed decline rate: {df[TARGET].mean():.4f}')
print(f'Frozen Week-4 CTR cutoff: {ctr_cutoff:.4f}%')
display(df[FEATURES + [TARGET]].describe().T.round(4))

In [ ]:
def precision_at_k(frame, score_col, k):
    ranked = frame.sort_values([score_col,'march_impressions','march_clicks'], ascending=[False,False,False])
    return float(ranked.head(min(k,len(ranked)))[TARGET].mean())

def make_model():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('logreg', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])

# BEFORE: reproduce the Week-5 stratified row split.
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.20, random_state=RANDOM_STATE, stratify=df[TARGET])
before_train, before_test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()
before_model = make_model().fit(before_train[FEATURES], before_train[TARGET])
before_test['model_probability'] = before_model.predict_proba(before_test[FEATURES])[:,1]

# AFTER: client-grouped, stratified out-of-fold predictions.
after = df.copy()
after['model_probability'] = np.nan
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr, va) in enumerate(cv.split(df[FEATURES], df[TARGET], groups=df[GROUP]), start=1):
    m = make_model().fit(df.iloc[tr][FEATURES], df.iloc[tr][TARGET])
    after.iloc[va, after.columns.get_loc('model_probability')] = m.predict_proba(df.iloc[va][FEATURES])[:,1]
    overlap = set(df.iloc[tr][GROUP]) & set(df.iloc[va][GROUP])
    assert not overlap, f'Client leakage in fold {fold}'

# Frozen Week-4 baseline is applied without fitting on the validation labels.
for frame in (before_test, after):
    frame['baseline_high_volume'] = frame['march_impressions'] >= 500
    frame['baseline_ctr_fix'] = (frame['baseline_high_volume'] & frame['march_avg_position'].between(4,20,inclusive='both') & frame['march_ctr_pct'].notna() & (frame['march_ctr_pct'] < ctr_cutoff))
    frame['baseline_score'] = np.select([frame['baseline_high_volume'] & frame['baseline_ctr_fix'], frame['baseline_high_volume']], [5,3], default=0).astype(float)

before_rows = []
after_rows = []
for k in KS:
    before_rows.append({'evaluation':'Before — random row split','k':k,'precision_at_k':precision_at_k(before_test,'model_probability',k)})
    after_rows.append({'evaluation':'After — client-grouped OOF','k':k,'precision_at_k':precision_at_k(after,'model_probability',k)})
comparison = pd.DataFrame(before_rows + after_rows)
comparison['precision_pct'] = (100*comparison['precision_at_k']).round(2)
display(comparison)

secondary = pd.DataFrame([
    {'evaluation':'Before — random row split','ROC_AUC':roc_auc_score(before_test[TARGET],before_test['model_probability']),'Average_Precision':average_precision_score(before_test[TARGET],before_test['model_probability'])},
    {'evaluation':'After — client-grouped OOF','ROC_AUC':roc_auc_score(after[TARGET],after['model_probability']),'Average_Precision':average_precision_score(after[TARGET],after['model_probability'])},
])
display(secondary.round(4))
print('Largest client overlap in BEFORE split:', len(set(before_train[GROUP]) & set(before_test[GROUP])))


### Before/after interpretation

The important comparison is not whether the grouped result is larger. The question is whether the Week-5 estimate changes when the same client is prevented from appearing in both training and validation. A drop is evidence that the original row split may have benefited from client overlap; a similar result is evidence that the measured ranking signal is more stable across unseen clients. Neither result proves production performance.

## 3. Leakage audit

The final feature set is checked against four leakage routes: explicit outcome/future columns, feature construction timing, identifiers/repeated entities, and population/label construction.

In [ ]:
leakage_checks = []

future_or_label = {'april_impressions','future_decline_label','trend_direction','trend_pct','label','target'}
bad_features = sorted(set(FEATURES) & future_or_label)
leakage_checks.append({'check':'Future/label-derived columns absent from features','status':'PASS' if not bad_features else 'FAIL','detail':str(bad_features)})

id_features = sorted(set(FEATURES) & {'client_hash_id','content_hash_id'})
leakage_checks.append({'check':'Client/content IDs excluded from predictive features','status':'PASS' if not id_features else 'FAIL','detail':str(id_features)})

march_only = all(c.startswith('march_') for c in FEATURES)
leakage_checks.append({'check':'Predictive features are March-only','status':'PASS' if march_only else 'FAIL','detail':str(FEATURES)})

label_uses_april = 'april_impressions' in df.columns and df[TARGET].notna().all()
leakage_checks.append({'check':'April appears only in the constructed outcome, not X','status':'PASS' if label_uses_april and 'april_impressions' not in FEATURES else 'FAIL','detail':'Label is defined from April impressions versus March impressions'})

key_unique = not df.duplicated(['client_hash_id','content_hash_id']).any()
leakage_checks.append({'check':'One row per client/content evaluation key','status':'PASS' if key_unique else 'FAIL','detail':'one-to-one March/April merge asserted'})

group_count = df[GROUP].nunique()
leakage_checks.append({'check':'Grouped validation has multiple clients','status':'PASS' if group_count > 1 else 'FAIL','detail':f'{group_count:,} clients'})

leakage_audit = pd.DataFrame(leakage_checks)
display(leakage_audit)
assert (leakage_audit['status'] == 'PASS').all(), 'Leakage audit failed; inspect the table before submission.'

### Leakage conclusion

The audit does not claim that leakage is impossible. It documents the controls applied to this notebook: March-only predictive features, an April-derived evaluation label, exclusion of identifiers from X, one-to-one aggregation keys, and client-grouped validation. Any future feature added to the pipeline must be checked against the same decision-time rule.

## 4. Real failure examples

The examples below are deliberately anonymized: only model inputs, score, and outcome are shown. They are intended to show where the ranking makes mistakes, not to imply why a specific client/page changed.

In [ ]:
# Highest-confidence false positives and false negatives under the honest grouped evaluation.
fp = after[(after['model_probability'] >= 0.5) & (after[TARGET] == 0)].sort_values('model_probability', ascending=False).head(5)
fn = after[(after['model_probability'] < 0.5) & (after[TARGET] == 1)].sort_values('model_probability', ascending=True).head(5)
show_cols = FEATURES + ['model_probability', TARGET]
print('False positives — high model score, no observed decline:')
display(fp[show_cols].round(6))
print('False negatives — low model score, observed decline:')
display(fn[show_cols].round(6))

print('Failure counts:', {'false_positives': int(((after['model_probability'] >= 0.5) & (after[TARGET] == 0)).sum()), 'false_negatives': int(((after['model_probability'] < 0.5) & (after[TARGET] == 1)).sum())})


### Failure interpretation

False positives show where March signals looked concerning but the measured April outcome did not cross the decline threshold. False negatives show where the March signals looked comparatively healthy but the measured outcome still declined. These examples demonstrate uncertainty and boundary cases; they do not establish causal explanations for the underlying SEO changes.

## 5. Claim rewrite

### Original Week-5-style claim
> The logistic regression model improves ranking quality and can predict which pages will decline.

### Safer claim
The model **measured higher/lower Precision@K under the original random row split than under the client-grouped evaluation** for the tested March→April population. The grouped result is the more conservative estimate for unseen-client generalization. The model is therefore best described as **decision-support for prioritization**, not as a guarantee that an individual page will decline or that a refresh will improve performance.

The exact direction and values are printed from the executed comparison table above; no metric is hard-coded into this narrative.

In [ ]:
# Generate a public-safe claim from the actual executed results.
before50 = float(comparison.loc[(comparison.evaluation.str.startswith('Before')) & (comparison.k == 50), 'precision_pct'].iloc[0])
after50 = float(comparison.loc[(comparison.evaluation.str.startswith('After')) & (comparison.k == 50), 'precision_pct'].iloc[0])
delta50 = after50 - before50
direction = 'higher' if delta50 > 0 else 'lower' if delta50 < 0 else 'similar'
print(f'Observed Precision@50 was {before50:.2f}% before and {after50:.2f}% after client grouping (difference: {delta50:+.2f} percentage points).')
print(f'Safe claim: On this tested March→April population, the measured Precision@50 was {direction} under client-grouped validation than under the original random row split. This result is directional evidence for decision-support, not a guarantee of future page-level outcomes or business impact.')

## Self-check

- [ ] Every section is filled — methodology reasoning AND executable evidence
- [ ] Runtime → Run all completes without errors after adding `HF_TOKEN` in Colab Secrets
- [ ] Before/after comparison uses the established March→April lane and model family
- [ ] Honest split is grouped by `client_hash_id`; validation folds have zero client overlap
- [ ] April/future/label-derived fields are excluded from predictive features
- [ ] IDs are used for grouping/audit only
- [ ] Failure examples contain no client names, URLs, or private queries
- [ ] Claims use observed/measured/directional/decision-support language
- [ ] The executed notebook is committed under `work/notebooks/w06_validation_audit.ipynb`